# 6 RL - 强化学习后训练

基于 05_reward 设定的种种奖励规则可以完成整个训练的最后一个环节，通过优化奖励目标的方式来对模型进一步调优。GPT 类的语言模型在这个阶段的训练手段有很多种，鉴于我们的模型规模很小，问题也相对容易，因此在这个阶段使用最简单的训练策略——**REINFORCE 算法**。

## 训练策略

**REINFORCE** 是一种经典的策略梯度算法，核心思想是：
1. 根据当前策略（模型）采样生成多个样本
2. 计算每个样本的奖励分数
3. 使用奖励信号调整策略，让高奖励的样本更容易生成

**目标函数**：
```
Loss = -Σ log P(token | context) × (reward - baseline)
```

其中：
- `log P(token | context)` 是模型生成该 token 的对数概率
- `reward` 是该样本的总奖励（来自 05_reward 中定义的三个目标）
- `baseline` 是奖励的均值，用于降低方差

## 奖励目标

我们使用在 05_reward 中定义的三个奖励目标：
1. **押韵检查** (30%)：句尾字的韵母是否一致
2. **格式检查** (30%)：诗词格式是否符合要求
3. **作者风格匹配** (40%)：使用训练好的分类器判断作者与诗词风格是否匹配

本 Notebook 只训练少量步骤，作为演示。完整训练请使用 `nanopoet/train_rl.py`。

In [1]:
import random
import torch
import torch.nn.functional as F

from nanopoet.common import CharTokenizer, encode_poem_prompt, decode_poem_str, CONTENT_START, CONTENT_END
from nanopoet.dataset import load_raw_data
from nanopoet.model import GPTLanguageModel
from nanopoet.reward import compute_reward, extract_format

# 初始化随机种子
random.seed(12345)
torch.manual_seed(12345)

# 设备选择
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f"使用设备: {device}")

# 加载数据（用于提取格式信息和作者列表）
data = load_raw_data("../raw")

# 初始化分词器
tokenizer = CharTokenizer("".join(["".join(list(d.values())) for d in data]))
print(f"词表大小: {tokenizer.vocab_size}")

使用设备: mps
词表大小: 11868


## 加载模型和奖励分类器

In [2]:
from nanopoet.common import filter_by_author, update_poem_author, AUTHOR_S, STYLE_T, STYLE_S

# 准备格式数据（用于格式检查）
filtered_data = [update_poem_author(d) for d in data if filter_by_author(d)]
format_data = extract_format(filtered_data)
print(f"提取了 {len(format_data)} 种诗词格式")

# 定义训练用的作者和风格列表
train_authors = AUTHOR_S
train_styles = STYLE_T + STYLE_S
print(f"训练作者: {len(train_authors)} 人")
print(f"训练风格: {len(train_styles)} 种")

提取了 145 种诗词格式
训练作者: 15 人
训练风格: 10 种


In [3]:
# 初始化模型结构
block_size = 256
model = GPTLanguageModel(
    vocab_size=tokenizer.vocab_size,
    emb_size=256,
    block_size=block_size,
    layer_num=8,
    head_num=8,
    dropout=0.0,  # RL 训练阶段不需要 dropout
).to(device)

# 加载 SFT 模型
sft_model_path = "./output/04_sft_model.pt"
print(f"加载 SFT 模型: {sft_model_path}")
sft_state = torch.load(sft_model_path, map_location=device)
model.load_state_dict(sft_state, strict=True)
print("✓ SFT 模型加载成功")

# 统计参数
total_params = sum(p.numel() for p in model.parameters())
print(f"模型参数量: {total_params:,}")

加载 SFT 模型: ./output/04_sft_model.pt
✓ SFT 模型加载成功
模型参数量: 12,466,268


In [4]:
# 加载奖励分类器（用于作者风格匹配）
from nanopoet.train_reward import BinaryClassifier

# 创建分类器结构（与训练时相同）
reward_gpt = GPTLanguageModel(
    vocab_size=tokenizer.vocab_size,
    emb_size=256,
    block_size=block_size,
    layer_num=8,
    head_num=8,
    dropout=0.0,
).to(device)

classifier = BinaryClassifier(
    gpt_model=reward_gpt,
    freeze_base=True,
    num_bidirectional_layers=2
).to(device)

# 加载训练好的分类器权重
classifier_path = "./output/05_author_classifier.pt"
print(f"加载奖励分类器: {classifier_path}")
classifier_checkpoint = torch.load(classifier_path, map_location=device)
classifier.load_state_dict(classifier_checkpoint['classifier_state_dict'])
classifier.eval()  # 分类器始终保持评估模式
print("✓ 奖励分类器加载成功")

加载奖励分类器: ./output/05_author_classifier.pt
✓ 奖励分类器加载成功


## RL 训练辅助函数

In [5]:
def generate_rl_prompt():
    """
    生成 RL 训练的提示
    随机选择作者和风格，50% 概率包含每个元数据
    """
    prompt_dict = {}
    
    # 50% 概率包含作者
    if random.random() > 0.5:
        prompt_dict['author'] = random.choice(train_authors)
    
    # 50% 概率包含风格
    if random.random() > 0.5:
        prompt_dict['style'] = random.choice(train_styles)
    
    return prompt_dict


@torch.no_grad()
def sample_from_model(model, prompt_str, num_samples, max_new_tokens, temperature=1.0, top_k=50):
    """
    从模型采样生成多个样本
    
    Args:
        model: GPT 模型
        prompt_str: 提示字符串
        num_samples: 采样数量
        max_new_tokens: 最大生成 token 数
        temperature: 采样温度
        top_k: Top-K 采样
    
    Returns:
        List[str]: 生成的样本列表
    """
    model.eval()
    
    # 编码提示
    prompt_tokens = tokenizer.encode(prompt_str)
    prefix_length = len(prompt_tokens)
    
    # 结束 token
    end_token_id = tokenizer.encode(CONTENT_END)[0]
    
    # 生成样本
    generated_samples = []
    
    for _ in range(num_samples):
        # 准备输入
        tokens = torch.tensor([prompt_tokens], dtype=torch.long, device=device)
        
        # 生成序列
        for _ in range(max_new_tokens):
            # 截断到 block_size
            input_tokens = tokens[:, -block_size:] if tokens.size(1) > block_size else tokens
            
            # 前向传播
            logits, _ = model(input_tokens)
            logits = logits[:, -1, :] / temperature
            
            # Top-K 采样
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
            
            # 采样
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            
            # 拼接
            tokens = torch.cat([tokens, next_token], dim=1)
            
            # 检查是否生成了结束 token
            if next_token.item() == end_token_id:
                break
        
        # 解码
        sample_text = tokenizer.decode(tokens[0].tolist())
        generated_samples.append(sample_text)
    
    return generated_samples


def compute_sample_reward(sample_str, classifier, tokenizer, format_data, device):
    """
    计算单个样本的奖励
    
    Args:
        sample_str: 生成的样本字符串（完整的编码格式）
        classifier: 奖励分类器
        tokenizer: 分词器
        format_data: 格式数据
        device: 设备
    
    Returns:
        float: 总奖励值 (0-1)
    """
    try:
        # 使用 compute_reward 计算奖励
        reward_result = compute_reward(
            poem_str=sample_str,
            classifier=classifier,
            tokenizer=tokenizer,
            format_data=format_data,
            device=device
        )
        return reward_result['total_reward']
    except Exception as e:
        # 如果计算失败，返回最低奖励
        print(f"计算奖励时出错: {e}")
        return 0.0


print("✓ RL 训练辅助函数定义完成")

✓ RL 训练辅助函数定义完成


## 测试生成和奖励计算

In [6]:
# 测试：生成样本并计算奖励
print("测试样本生成和奖励计算...\n")

# 生成一个测试提示
test_prompt_dict = {'author': '李白', 'style': '七言绝句'}
test_prompt_str = encode_poem_prompt(**test_prompt_dict) + CONTENT_START
print(f"提示: {test_prompt_str}")

# 生成样本
test_samples = sample_from_model(
    model=model,
    prompt_str=test_prompt_str,
    num_samples=3,
    max_new_tokens=100,
    temperature=1.0,
    top_k=50
)

# 计算奖励
for i, sample in enumerate(test_samples, 1):
    reward = compute_sample_reward(sample, classifier, tokenizer, format_data, device)
    poem_dict = decode_poem_str(sample)
    content = poem_dict.get('content', '(无内容)')
    
    print(f"\n样本 {i}:")
    print(f"  内容: {content}")
    print(f"  奖励: {reward:.4f}")

测试样本生成和奖励计算...

提示: BA李白aS七言绝句sC

样本 1:
  内容: 此夜日不三三中年，天日時自見如誰如夜前春。君，春何家生有心老花。
  奖励: 0.4967

样本 2:
  内容: 。一二年日上春，古有秋爲君水，老行歸。春如誰。三歸。誰一，得。
  奖励: 0.4935

样本 3:
  内容: 日事時，其年高雲不天一其今不歸。
  奖励: 0.1959


## REINFORCE 训练

In [7]:
# RL 训练配置
NUM_SAMPLES_PER_PROMPT = 4  # 每个提示采样多少个样本
NUM_STEPS = 20              # 训练步数（Notebook 演示用，实际训练需要更多）
LEARNING_RATE = 1e-5        # 学习率（较小，避免过度更新）
TEMPERATURE = 1.0           # 采样温度
TOP_K = 50                  # Top-K 采样
MAX_NEW_TOKENS = 100        # 最大生成 token 数

# 初始化优化器
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

print("开始 RL 训练...\n")
print(f"训练配置:")
print(f"  训练步数: {NUM_STEPS}")
print(f"  每步采样数: {NUM_SAMPLES_PER_PROMPT}")
print(f"  学习率: {LEARNING_RATE:.2e}")
print(f"  采样温度: {TEMPERATURE}")
print(f"  Top-K: {TOP_K}")
print()

# 记录奖励历史
reward_history = []

for step in range(NUM_STEPS):
    # ========== 步骤 1: 生成提示 ==========
    prompt_dict = generate_rl_prompt()
    prompt_str = encode_poem_prompt(**prompt_dict) + CONTENT_START
    
    # ========== 步骤 2: 采样生成样本 ==========
    samples = sample_from_model(
        model=model,
        prompt_str=prompt_str,
        num_samples=NUM_SAMPLES_PER_PROMPT,
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=TEMPERATURE,
        top_k=TOP_K
    )
    
    # ========== 步骤 3: 计算奖励 ==========
    rewards = []
    for sample in samples:
        reward = compute_sample_reward(sample, classifier, tokenizer, format_data, device)
        rewards.append(reward)
    
    rewards_tensor = torch.tensor(rewards, dtype=torch.float32, device=device)
    mean_reward = rewards_tensor.mean().item()
    
    # ========== 步骤 4: 计算优势（Advantage） ==========
    # 使用当前批次的均值作为 baseline
    advantages = rewards_tensor - rewards_tensor.mean()
    
    # ========== 步骤 5: 策略梯度更新 ==========
    model.train()
    optimizer.zero_grad()
    
    # 对每个样本计算策略梯度
    total_loss = 0.0
    
    for sample_idx, (sample, advantage) in enumerate(zip(samples, advantages)):
        # 编码样本
        sample_tokens = tokenizer.encode(sample)
        
        # 截断到 block_size + 1（输入和目标）
        if len(sample_tokens) > block_size + 1:
            sample_tokens = sample_tokens[:block_size + 1]
        
        # 准备输入和目标
        inputs = torch.tensor([sample_tokens[:-1]], dtype=torch.long, device=device)
        targets = torch.tensor([sample_tokens[1:]], dtype=torch.long, device=device)
        
        # 计算提示长度（不对提示部分计算梯度）
        prefix_length = len(tokenizer.encode(prompt_str))
        
        # 创建 mask：只对生成的部分计算 loss
        mask = torch.zeros_like(targets[0])
        if prefix_length < len(targets[0]):
            mask[prefix_length:] = 1
        
        # 前向传播
        logits, _ = model(inputs)
        
        # 计算每个 token 的负对数似然
        log_probs = F.log_softmax(logits[0], dim=-1)
        token_log_probs = log_probs[range(len(targets[0])), targets[0]]
        
        # 只对生成的部分应用 advantage
        masked_log_probs = token_log_probs * mask
        
        # 策略梯度目标：log_prob * advantage
        # Loss = -Σ log_prob * advantage（最大化目标 = 最小化负目标）
        pg_loss = -(masked_log_probs * advantage).sum() / mask.sum().clamp(min=1)
        
        total_loss += pg_loss
    
    # 归一化 loss
    total_loss = total_loss / NUM_SAMPLES_PER_PROMPT
    
    # 反向传播
    total_loss.backward()
    
    # 梯度裁剪
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    
    # 更新参数
    optimizer.step()
    
    # ========== 记录和输出 ==========
    reward_history.append(mean_reward)
    
    # 计算移动平均
    window_size = min(10, len(reward_history))
    moving_avg = sum(reward_history[-window_size:]) / window_size
    
    print(f"步数 {step + 1:3d}/{NUM_STEPS} | "
          f"奖励: {mean_reward:.4f} | "
          f"移动平均: {moving_avg:.4f} | "
          f"Loss: {total_loss.item():.4f}")
    
    # 每 5 步展示一个样本
    if (step + 1) % 5 == 0:
        best_idx = rewards.index(max(rewards))
        best_sample = samples[best_idx]
        best_reward = rewards[best_idx]
        
        poem_dict = decode_poem_str(best_sample)
        print(f"  最佳样本 (奖励={best_reward:.4f}):")
        print(f"    作者: {poem_dict.get('author', '(无)')}")
        print(f"    风格: {poem_dict.get('style', '(无)')}")
        print(f"    内容: {poem_dict.get('content', '(无内容)')}")
        print()

print("\n✓ RL 训练完成！")

开始 RL 训练...

训练配置:
  训练步数: 20
  每步采样数: 4
  学习率: 1.00e-05
  采样温度: 1.0
  Top-K: 50

步数   1/20 | 奖励: 0.6625 | 移动平均: 0.6625 | Loss: -0.0020
步数   2/20 | 奖励: 0.2085 | 移动平均: 0.4355 | Loss: 0.0008
步数   3/20 | 奖励: 0.8125 | 移动平均: 0.5612 | Loss: -0.0424
步数   4/20 | 奖励: 0.5875 | 移动平均: 0.5677 | Loss: 0.0739
步数   5/20 | 奖励: 0.3746 | 移动平均: 0.5291 | Loss: 0.0012
  最佳样本 (奖励=0.5043):
    作者: 李清照
    风格: 五言律诗
    内容: 日日雲君，知時上今今如人心無花。君。風此山，雲下中，只君。一，老時，天。

步数   6/20 | 奖励: 0.4750 | 移动平均: 0.5201 | Loss: 0.0365
步数   7/20 | 奖励: 0.6711 | 移动平均: 0.5417 | Loss: -0.0048
步数   8/20 | 奖励: 0.5724 | 移动平均: 0.5455 | Loss: 0.0193
步数   9/20 | 奖励: 0.5875 | 移动平均: 0.5502 | Loss: 0.0200
步数  10/20 | 奖励: 0.8125 | 移动平均: 0.5764 | Loss: 0.0410
  最佳样本 (奖励=1.0000):
    作者: (无)
    风格: (无)
    内容: 李何。，知事，不更春。

步数  11/20 | 奖励: 0.3577 | 移动平均: 0.5459 | Loss: 0.0000
步数  12/20 | 奖励: 0.5125 | 移动平均: 0.5763 | Loss: -0.0050
步数  13/20 | 奖励: 0.6387 | 移动平均: 0.5590 | Loss: 0.0015
步数  14/20 | 奖励: 0.3186 | 移动平均: 0.5321 | Loss: -0.0051
步数  15/20 | 奖励:

### 关于移动平均

**移动平均**（Moving Average）是用于平滑数据波动的统计指标。在 RL 训练中，由于采样的随机性，单步奖励会有较大波动，难以判断训练趋势。

**为什么需要移动平均？**
- 单步奖励受采样随机性影响，波动较大
- 移动平均能够过滤短期波动，反映整体趋势
- 帮助我们判断训练是否在改进

**如何计算？**
```python
window_size = 50  # 窗口大小（本 Notebook 用 10 步演示）
moving_avg = sum(reward_history[-window_size:]) / window_size
```

**如何解读？**
- 关注**移动平均**的趋势，而非单步奖励的波动
- 如果移动平均持续上升 ↑，说明训练有效
- 如果移动平均趋于平稳 →，说明模型收敛
- 如果移动平均持续下降 ↓，需要检查训练配置

## 训练前后对比

In [8]:
# 对比训练前后的生成质量
print("训练后生成样本对比...\n")

# 重新加载 SFT 模型（训练前）
model_before = GPTLanguageModel(
    vocab_size=tokenizer.vocab_size,
    emb_size=256,
    block_size=block_size,
    layer_num=8,
    head_num=8,
    dropout=0.0,
).to(device)
model_before.load_state_dict(sft_state)
model_before.eval()

# 当前模型（训练后）
model.eval()

# 生成对比样本
test_cases = [
    {'author': '李白', 'style': '五言绝句'},
    {'author': '杜甫', 'style': '七言律诗'},
    {'author': '苏轼', 'style': '水调歌头'},
]

for test_case in test_cases:
    prompt_str = encode_poem_prompt(**test_case) + CONTENT_START
    
    print(f"提示: 作者={test_case.get('author', '无')}, 风格={test_case.get('style', '无')}")
    print()
    
    # 训练前
    samples_before = sample_from_model(model_before, prompt_str, num_samples=1, max_new_tokens=100)
    reward_before = compute_sample_reward(samples_before[0], classifier, tokenizer, format_data, device)
    poem_before = decode_poem_str(samples_before[0])
    
    print(f"  训练前 (奖励={reward_before:.4f}):")
    print(f"    {poem_before.get('content', '(无内容)')}")
    print()
    
    # 训练后
    samples_after = sample_from_model(model, prompt_str, num_samples=1, max_new_tokens=100)
    reward_after = compute_sample_reward(samples_after[0], classifier, tokenizer, format_data, device)
    poem_after = decode_poem_str(samples_after[0])
    
    print(f"  训练后 (奖励={reward_after:.4f}):")
    print(f"    {poem_after.get('content', '(无内容)')}")
    
    improvement = reward_after - reward_before
    print(f"  改进: {improvement:+.4f}")
    print("=" * 70)
    print()

训练后生成样本对比...

提示: 作者=李白, 风格=五言绝句

  训练前 (奖励=0.4931):
    平此。年此月，秋來有江見，一年。

  训练后 (奖励=0.3577):
    長心秋有清清，春。天，日人風。江，年人山風今來。
  改进: -0.1353

提示: 作者=杜甫, 风格=七言律诗

  训练前 (奖励=0.3667):
    獨，一此人人春自水在在水人日，相人風雨花。不，長。水有來。風，誰，不如長一老，水。。

  训练后 (奖励=0.5137):
    月未雨，長何雲風江老。何，三月水有歸自不君去，爲今人無君雲。風江君家此自生無知老，年如知，高風人。老君。
  改进: +0.1469

提示: 作者=苏轼, 风格=水调歌头

  训练前 (奖励=0.3583):
    十一子門家，天不更可誰。獨此無江今如相何無雨，時。自來。得，雲自風有天。

  训练后 (奖励=0.3527):
    不相不來是三十風歸春時，如一風。夜。不今高相，日事，獨得去。年處得，有千如無。
  改进: -0.0055



## 保存模型

In [9]:
from pathlib import Path

# 保存 RL 训练后的模型
output_path = Path("./output/06_rl_model.pt")
output_path.parent.mkdir(parents=True, exist_ok=True)

torch.save({
    'model_state_dict': model.state_dict(),
    'final_reward': reward_history[-1] if reward_history else 0.0,
    'reward_history': reward_history,
}, output_path)

print(f"✓ RL 模型已保存到 {output_path}")
print(f"\n训练总结:")
print(f"  训练步数: {NUM_STEPS}")
print(f"  初始平均奖励: {reward_history[0]:.4f}" if reward_history else "  (无数据)")
print(f"  最终平均奖励: {reward_history[-1]:.4f}" if reward_history else "  (无数据)")
if len(reward_history) > 1:
    improvement = reward_history[-1] - reward_history[0]
    print(f"  总体改进: {improvement:+.4f}")

✓ RL 模型已保存到 output/06_rl_model.pt

训练总结:
  训练步数: 20
  初始平均奖励: 0.6625
  最终平均奖励: 0.6197
  总体改进: -0.0428
